# Section 3: Retrieval & Knowledge Patterns
## AI-Native Software Architecture | O'Reilly Course

In Section 2, we improved how the model receives instructions and how its output is structured.

The model still does not have access to current company policy or relevant customer history.

In this exercise, we will compare the same request across three variations:

1. No retrieval
2. Retrieved policy evidence
3. Retrieved policy evidence plus relevant memory

### Observe

- Does the recommendation become grounded in policy?
- Does the response cite supporting evidence?
- Does memory add useful context?
- Does additional context introduce distraction or risk?

In [ ]:
import os
import support_utils.llm_client as llm_client

from support_utils import (
    call_llm,
    primary_issue,
    knowledge_base,
    enhanced_retrieve,
    nshot_support_prompt,
    rag_support_prompt,
    conversation_memory,
    remember,
    get_recent_memory,
    format_memory,
    memory_aware_rag_prompt,
)

In [ ]:
# False: credential-free dummy LLM
# True: Vertex AI when configured, otherwise OpenAI
USE_REAL_LLM = True
llm_client.USE_REAL_LLM = USE_REAL_LLM

if not USE_REAL_LLM:
    provider = "Dummy LLM"
elif llm_client.gemini_client is not None:
    provider = f"Vertex AI ({llm_client.GEMINI_MODEL})"
elif os.getenv("OPENAI_API_KEY"):
    provider = f"OpenAI ({llm_client.OPENAI_MODEL})"
else:
    provider = "No real LLM configured"

print(f"Provider: {provider}")
print(f"Customer issue: {primary_issue}")

## Exercise Setup

For this workshop, `knowledge_base` is a small in-memory collection of company policies.

It stands in for an external knowledge source so that we can focus on evidence selection and generation without requiring a separate search service.

In [ ]:
print("=== Available policy documents ===")

for doc in knowledge_base:
    print(f"\n📄 {doc['doc_id']}: {doc['title']}")
    print(f"Freshness: {doc['freshness']}")
    print(f"Trust score: {doc['trust_score']}")
    print(f"Policy: {doc['text']}")

## Variation 1: No Retrieval

First, run the structured support flow without company policy.

The model receives instructions and examples, but no authoritative evidence about the refund policy.

In [ ]:
no_retrieval_output = call_llm(
    nshot_support_prompt(primary_issue),
    temperature=0.2,
    force_json=True,
)

print("=== No retrieval ===")
print(no_retrieval_output)

## Variation 2: Add Retrieved Policy Evidence

Now retrieve the most relevant policy for the same customer request.

The retrieval system decides which evidence enters the model context.

In [ ]:
selected_docs = enhanced_retrieve(
    primary_issue,
    top_k=1,
)

print("=== Selected evidence ===")

for doc in selected_docs:
    score = doc.get(
        "retrieval_score",
        doc.get("final_score", "not available"),
    )

    print(f"\n📄 {doc['doc_id']}: {doc['title']}")
    print(f"Retrieval score: {score}")
    print(f"Freshness: {doc['freshness']}")
    print(f"Trust score: {doc['trust_score']}")
    print(f"Policy: {doc['text']}")

### Before Generation, Check the Evidence

The expected source for this request is:

`POLICY_REFUND_DUPLICATE`

If the wrong source is selected, the generated answer may be wrong even if the model follows its instructions perfectly.

In [ ]:
expected_doc_id = "POLICY_REFUND_DUPLICATE"
selected_doc_ids = [doc["doc_id"] for doc in selected_docs]

print("Selected document IDs:", selected_doc_ids)
print("Expected evidence selected:", expected_doc_id in selected_doc_ids)

In [ ]:
with_retrieval_output = call_llm(
    rag_support_prompt(primary_issue, selected_docs),
    temperature=0.2,
    force_json=True,
)

print("=== With retrieved policy evidence ===")
print(with_retrieval_output)

## Variation 3: Add Relevant Memory

Retrieval provides company knowledge. Memory provides relevant history from prior interactions.

We will add one prior interaction for the same customer, then run the request again with both policy evidence and memory.

In [ ]:
# Keep the exercise repeatable when this cell is run more than once.
conversation_memory.clear()

remember(
    user_id="user_123",
    issue="I reported two subscription charges last week.",
    assistant_response=(
        "The customer confirmed the duplicate charges and was told "
        "that the refund decision requires human review."
    ),
)

recent_memory = get_recent_memory("user_123")

print("=== Relevant customer history ===")
print(format_memory(recent_memory))

In [ ]:
with_memory_output = call_llm(
    memory_aware_rag_prompt(
        "user_123",
        primary_issue,
        selected_docs,
    ),
    temperature=0.2,
    force_json=True,
)

print("=== With retrieved evidence and memory ===")
print(with_memory_output)

## Compare the Three Variations

The customer request stayed the same. Only the context available to the model changed.

In [ ]:
exercise_3_outputs = {
    "1_no_retrieval": no_retrieval_output,
    "2_retrieved_policy": with_retrieval_output,
    "3_retrieved_policy_and_memory": with_memory_output,
}

for variation, output in exercise_3_outputs.items():
    print(f"\n=== {variation} ===")
    print(output)

## What Did We Observe?

Compare the three outputs:

- Which response is supported by company policy?
- Which response identifies the source it used?
- Did memory change the recommended action?
- Was the memory relevant, or was it unnecessary context?
- What could happen if retrieval selected the archived refund policy?

Retrieval quality constrains generation quality. Memory is useful only when it contributes relevant state.

## Section 3 Takeaway

Retrieval does not simply add more information. It selects the evidence available to the model.

Memory serves a different purpose by carrying relevant state across interactions.

A production system must decide:

- which evidence is eligible and relevant
- how much context is sufficient
- whether prior history is useful for the current request
- whether the generated response actually uses and cites its evidence

**Next:** Section 4: decisioning, fallback, and governance.